# EEG-Derived Image Classification for Drowsiness Experiments

A TensorFlow/Keras notebook that standardizes EEG-like arrays, converts signals into images, and trains an image classifier for awake/drowsy labels. Despite the original filename, the notebook does not implement facial landmarks, eye-aspect-ratio calculations, or real-time camera inference.

**Data and reproducibility:** The source arrays DrowD.npy and label.npy are not included. The notebook expects them under data/ and writes intermediate images/features under data/. It also contains a fixed-size reshape tied to the original dataset; update it for different sample counts. Outputs were removed, and no validated performance claim is made.

Install the dependencies listed in `requirements.txt` in a compatible Python environment before running the notebook. The saved outputs were cleared; training and evaluation cells have not been rerun as part of this cleanup.


# **Reading Raw Data and Saving into Numpy**

In [ ]:
# import os
# import scipy.io as sio
# mat_data = sio.loadmat('/workspace/Driver Drowsiness/Data/dataset.mat')
# mat_data

In [ ]:
# EEG_Sample = mat_data['EEGsample']
# SubIndex = mat_data['subindex']
# SubState = mat_data['substate']
# EEG_Sample.shape

In [ ]:
# import numpy as np
# Data = EEG_Sample
# Label = SubState
# np.save('/workspace/Driver Drowsiness/Data/Main_data/DrowD.npy', Data)
# np.save('/workspace/Driver Drowsiness/Data/Main_data/label.npy', Label)

# **Data Reading**

In [ ]:
import numpy as np
DrowD = np.load('data/DrowD.npy')
label = np.load('data/label.npy')
print("Shape of the Data: ", DrowD.shape)
print("Shape of the Label: ", label.shape)

In [ ]:
DrowD[0][0].shape

In [ ]:
for i in range(0,30):
  plt.plot(DrowD[1][i])
  plt.axis('off')
  plt.savefig('data/Original_Signal/'+'Drowsy-'+str(i)+'.png', bbox_inches='tight', pad_inches = 0)
  plt.close()
plt.show()

# **Pre-processing**

In [ ]:
Data = [lst[:] for lst in DrowD]

Ndata=[]
for i in  range(len(Data)):
    std=np.std(Data[i],axis=1)
    mean=np.mean(Data[i],axis=1)
    Data[i]=(Data[i].transpose()-mean.transpose()).transpose()
    Data[i]=(Data[i].transpose()/std.transpose()).transpose()
    Ndata.append(Data[i])

Ndata = np.array(Ndata)
# for i in range(len(Ndata)):
#     print(np.array(Ndata[i]).shape)

In [ ]:
for i in range(0,30):
  plt.plot(Ndata[1][i])
  plt.axis('off')
  plt.savefig('data/Standardized Plot/'+'Drowsy-'+str(i)+'.png', bbox_inches='tight', pad_inches = 0)
  plt.close()
plt.show()

# **Generating Image**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
path = 'data/Image_Data/0'
path1 = 'data/Image_Data/1'

for i in range(len(Ndata)): 
    if label[i][0] == 0:
        plt.pcolormesh(Ndata[i], cmap='viridis')
        plt.axis('off')
        filename = f'No_Drowsy_{i+1}.png'
        plt.savefig(path+'/'+filename, dpi=300, bbox_inches='tight')
    else:
        plt.pcolormesh(Ndata[i], cmap='viridis')
        plt.axis('off')
        filename = f'Drowsy_{i+1}.png'
        plt.savefig(path1+'/'+filename, dpi=300, bbox_inches='tight')

    # Close the current plot to avoid overlap with the next iteration
    plt.close()

print("All plots saved successfully.")

# **Reading the Image Dataset**

In [ ]:
import os,cv2
import numpy as np
import pandas as pd
from math import exp, sqrt
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels
import matplotlib.image as mpimg
from sklearn.utils import shuffle
from keras.utils import np_utils
import keras
import tensorflow as tf
import tensorflow.keras.backend as K
from keras import Input 
from keras.models import Model
from keras.layers.core import Dense, Dropout, Activation, Flatten
from keras.layers import  BatchNormalization, Concatenate, Add, DepthwiseConv2D, GlobalAveragePooling2D, SeparableConv2D
from tensorflow.keras.optimizers import SGD,RMSprop,Adam,Adagrad
from keras.layers.convolutional import Conv2D, MaxPooling2D

In [ ]:
data_path='data/Image_Data'
data_dir_list = os.listdir(data_path)
img_data_list=[]
labels = []
for dataset in data_dir_list:
    img_list=os.listdir(data_path+'/'+ dataset)
    print ('Loaded the images of dataset-'+'{}\n'.format(dataset))
    for img in img_list:
        if img[-4:] == '.png':
            input_img=cv2.imread(data_path + '/'+ dataset + '/'+ img )
#             img=cv2.cvtColor(input_img, cv2.COLOR_BGR2GRAY)
            img = cv2.resize(input_img,(224,224))
            labels.append(dataset)
            img_data_list.append(img)    
label=np.array(labels)
img_data = np.array(img_data_list)
img_data = img_data.astype('float32')
img_data = img_data/255
img_data.shape

# **Splitting the Data into Training and Testing**

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

num_classes = 2
Y = to_categorical(label, num_classes)
x,y = shuffle(img_data, Y, random_state = 42)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.1, random_state = 42)
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
import numpy as np
import pandas as pd
x = X_train.reshape(1819, -1)
np.savetxt('data/Feature_Files/Drow_X_train.csv', x, delimiter = ",")

q =  np.argmax(y_train, axis =1).reshape(1819, -1)
np.savetxt("data/Feature_Files/Drow_y_train.csv", q,  delimiter = ",")

In [ ]:
X_train = X_train.reshape(-1, 224, 224, 3)
X_test = X_test.reshape(-1, 224, 224, 3)

In [ ]:
from tensorflow.keras.layers import Conv2D, Activation, Multiply, Add, GlobalMaxPooling2D, Dense, Reshape, Permute, multiply, GlobalAveragePooling2D

def cbam_attention(inputs, reduction_ratio=8):
    # Channel attention
    x = inputs
    channel_axis = 1 if tf.keras.backend.image_data_format() == "channels_first" else -1
    channel = x.shape[channel_axis]
    shared_layer_one = Dense(channel // reduction_ratio, kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu', use_bias=True)
    shared_layer_two = Dense(channel, kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), use_bias=True)

    avg_pool = GlobalAveragePooling2D()(x)
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(x)
    max_pool = Reshape((1, 1, channel))(max_pool)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = Add()([avg_pool, max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    if tf.keras.backend.image_data_format() == "channels_first":
        cbam_feature = Permute((3, 1, 2))(cbam_feature)

    attention_feature = multiply([x, cbam_feature])

    # Spatial attention
    kernel_size = 7
    spatial_attention = Conv2D(1, (kernel_size, kernel_size), kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), padding='same', activation='sigmoid', use_bias=False)(attention_feature)

    if tf.keras.backend.image_data_format() == "channels_first":
        spatial_attention = Permute((3, 1, 2))(spatial_attention)

    attention_feature = multiply([attention_feature, spatial_attention])
    return attention_feature

In [ ]:
import keras
from keras.models import Model
from tensorflow.keras import regularizers
from keras.layers import Input, Conv2D, Add, Multiply, Concatenate, AveragePooling2D, Flatten, DepthwiseConv2D, BatchNormalization
from keras.layers import SeparableConv2D, MaxPooling2D, Dense, GlobalAveragePooling2D, Activation
from tensorflow.keras.applications import ResNet50, VGG16, MobileNet

base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:12]:
    layer.trainable = False
x = base_model.output
x1 = Conv2D(256, (3, 3),  dilation_rate = 1, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x1 = BatchNormalization()(x1)
x2 = Conv2D(256, (3, 3),  dilation_rate = 3, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x2 = BatchNormalization()(x2)
x3 = Conv2D(256, (3, 3),  dilation_rate = 5, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x3 = BatchNormalization()(x3)

A1 = Add()([x1, x2])
M1 = Multiply()([x2, x3])

x4 = Conv2D(256, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(A1)
x4 = BatchNormalization()(x4)
g1 = GlobalAveragePooling2D()(x4)
d1 = Dense(1, activation = 'sigmoid')(g1)
M3 = Multiply()([d1, x1])

x5 = Activation('softmax')(M1)
M2 = Multiply()([x3, x5])

C1 = Concatenate()([M2, M3])
x5 = Conv2D(256, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(C1)
C2 = Concatenate()([x5, x])
x = cbam_attention(C2)
x = Add()([x, C2])

x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
output = Dense(2, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer="adam", loss= 'categorical_crossentropy', metrics="accuracy")
model.summary()

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(
                              patience=30,
                              min_delta=0.001,
                              monitor="val_accuracy",
                              restore_best_weights=True
                              )
# Define the model checkpoint callback to save the best weights
checkpoint = ModelCheckpoint('data/Weight_Files/'+'-{epoch:02d}.h5', monitor='val_accuracy', save_best_only=True)

history = model.fit( X_train, y_train, batch_size = 32, epochs = 100,
                    validation_data = (X_test , y_test), callbacks=[early_stopping, checkpoint], verbose =1)

In [ ]:
model_size = model.count_params() * 4 / (1024 * 1024) 
print("Model Size (MB):", model_size)

In [ ]:
import keras
model = keras.models.load_model('data/Weight_Files/-32.h5')

In [ ]:
from keras.models import Model
model = model
layer_name = 'dense_85'
intermediate_layer_model = Model(inputs = model.input, outputs = model.get_layer(layer_name).output)
intermediate_output1 = intermediate_layer_model.predict([X_train])

In [ ]:
import numpy as np
import pandas as pd
np.savetxt("data/Feature_Files/Drow_Proposed.csv", intermediate_output1, delimiter = ",")

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Predicting the Test set results
pred = model.predict(X_test)
print("Y_pred:", pred)
print("*****************")
y_pred = np.argmax(pred, axis = 1)
y_true = np.argmax(y_test, axis = 1)

In [ ]:
#Making the Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
target_names = ['Awake', 'Drowsy']
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.25)
p = sns.heatmap(pd.DataFrame(cm), annot=True,xticklabels=target_names, yticklabels=target_names, cmap="YlGnBu" ,fmt='g')
# plt.title('Confusion matrix', y=1.1)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.savefig('data/Feature_Files/Drowsy_CM.png')

In [ ]:
#Making the Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
target_names = ['Awake', 'Drowsy']
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.25)
cm = cm.astype('float')/cm.sum(axis=1)[:, np.newaxis]
p = sns.heatmap(pd.DataFrame(cm), annot=True,xticklabels=target_names, yticklabels=target_names, cmap="YlGnBu" ,fmt='.3f')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.savefig('data/Feature_Files/Drowsy_NCM.png')

In [ ]:
#import classification_report
from sklearn.metrics import classification_report
print(classification_report(y_true,y_pred, target_names = target_names))

In [ ]:
from sklearn.metrics import roc_curve, auc
# sns.set(font_scale=1.25)
y_pred_proba = model.predict(X_test)
fpr, tpr, thresholds = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)
plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr, color='green', label='AUC (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate', fontsize = 15)
plt.ylabel('True Positive Rate', fontsize = 15)
plt.legend(loc="lower right")
plt.title('ROC curve')
plt.grid(True)
plt.axis(True)
plt.savefig('data/Feature_Files/Drowsy_ROC.png')
plt.show()

# **Feature Visualization**

In [ ]:
import pandas as pd
df = pd.read_csv('data/Feature_Files/Drow_X_train.csv', header = None)
df.head()

In [ ]:
df1 = pd.read_csv("data/Feature_Files/Drow_y_train.csv", header = None,  delimiter = ",")
df1.head()

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
X = np.array(df)
X_embedded = TSNE(n_components=2).fit_transform(X)

In [ ]:
np.save('data/Feature_Files/X_train_embedded.npy', X_embedded)